# 🛡️ BlockSure v2.0 — Google Colab Interactive Demonstration
### Blockchain-Based File Integrity Verification with Provenance and Authorized Updates

- **Repository**: [https://github.com/vaishnavir2626/blockchain_file_integrity](https://github.com/vaishnavir2626/blockchain_file_integrity)
- **Live Production Web App**: [https://singular-tartufo-502127.netlify.app](https://singular-tartufo-502127.netlify.app)

This interactive notebook demonstrates the core cryptographic and smart contract features of BlockSure directly inside Google Colab:
1. **Environment Setup & Codebase Cloning**
2. **Automated Smart Contract Test Execution (33 Tests Passing)**
3. **Empirical Benchmark Evaluation (Accuracy, Throughput, Gas, Latency)**
4. **Interactive File Integrity & Tamper Detection Demo**
5. **Benchmark Visualizations (Matplotlib Plots)**

--- 
## 1. Environment Setup & Repository Cloning

In [ ]:
# Clone the official BlockSure GitHub repository
!rm -rf blockchain_file_integrity
!git clone https://github.com/vaishnavir2626/blockchain_file_integrity.git
%cd blockchain_file_integrity

# Check Node.js and npm versions
!node -v
!npm -v

# Install project dependencies
!npm install

--- 
## 2. Compile Smart Contracts & Run 33 Automated Tests
Verifies all 10 core requirements specified in PRD Section 8:
- Initial registration & verified status
- Tamper detection & renaming invariant test
- Authorized updates & lineage preservation
- Collaborator authorization & revocation
- Gas consumption & transaction inspection

In [ ]:
# Compile Solidity smart contract (FileIntegrityRegistry.sol)
!npx hardhat compile

# Run full automated test suite
!npx hardhat test

--- 
## 3. Run Empirical Experimental Evaluation Benchmark
Measures:
1. Hash-Detection Accuracy (100%)
2. SHA-256 Hashing Throughput (MB/s) and Latency
3. Smart Contract Gas Consumption Profile
4. Query and Transaction Confirmation Latency

In [ ]:
!node scripts/experimental_evaluation.js

--- 
## 4. Interactive File Integrity Verification Demo in Python
Demonstrates the zero-byte tamper detection mechanism using Python hashlib and simulated on-chain state verification.

In [ ]:
import hashlib
import os

# Load sample files
with open('sample_files/document_v1.txt', 'rb') as f:
    content_v1 = f.read()

with open('sample_files/document_v1_tampered.txt', 'rb') as f:
    content_tampered = f.read()

with open('sample_files/document_v2.txt', 'rb') as f:
    content_v2 = f.read()

# Calculate SHA-256 digests
hash_v1 = '0x' + hashlib.sha256(content_v1).hexdigest()
hash_tampered = '0x' + hashlib.sha256(content_tampered).hexdigest()
hash_v2 = '0x' + hashlib.sha256(content_v2).hexdigest()

# Simulated on-chain record for 'SPEC-SEC-2026'
on_chain_record = {
    'fileId': 'SPEC-SEC-2026',
    'owner': '0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266',
    'currentVersion': 2,
    'versions': {
        1: {'hash': hash_v1, 'name': 'document_v1.txt', 'previousHash': '0x' + '0'*64},
        2: {'hash': hash_v2, 'name': 'document_v2.txt', 'previousHash': hash_v1}
    }
}

def verify_file(file_bytes, filename):
    calc_hash = '0x' + hashlib.sha256(file_bytes).hexdigest()
    latest_hash = on_chain_record['versions'][on_chain_record['currentVersion']]['hash']
    
    print('='*70)
    print(f'Verifying: {filename}')
    print(f'Calculated SHA-256: {calc_hash}')
    print(f'On-Chain v2 Hash:  {latest_hash}')
    
    if calc_hash == latest_hash:
        print('STATUS: 🛡️ [VERIFIED] — 100% Cryptographic Match with Version 2!')
    elif any(calc_hash == v['hash'] for v in on_chain_record['versions'].values()):
        matched_ver = [ver for ver, v in on_chain_record['versions'].items() if v['hash'] == calc_hash][0]
        print(f'STATUS: ⚠️ [HISTORICAL VERSION] — Matches supersceded Version {matched_ver}.')
    else:
        print('STATUS: 🚨 [CRITICAL TAMPERING DETECTED] — Mismatch! Content modified or forged.')
    print('='*70 + '\n')

# Test 1: Authentic v2
verify_file(content_v2, 'document_v2.txt')

# Test 2: Tampered v1 copy
verify_file(content_tampered, 'document_v1_tampered.txt')

# Test 3: Superseded v1
verify_file(content_v1, 'document_v1.txt')

# Test 4: Renaming Invariant Test (same bytes as v2, different filename)
verify_file(content_v2, 'renamed_document_v2_new_name.txt')

--- 
## 5. Visualizing Experimental Benchmark Results
Plots hashing throughput across payload sizes, gas consumption by smart contract operation, and verification latency.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set dark modern style
plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Hashing Throughput (MB/s)
payload_sizes = ['1 KB', '64 KB', '512 KB', '1 MB', '5 MB', '10 MB']
throughputs = [6.27, 113.68, 159.49, 153.51, 166.52, 160.05]

bars1 = ax1.bar(payload_sizes, throughputs, color='#06b6d4', edgecolor='#22d3ee', alpha=0.85)
ax1.set_title('SHA-256 Hashing Throughput by Payload Size', fontsize=12, fontweight='bold', pad=12)
ax1.set_ylabel('Throughput (MB/s)', fontsize=10)
ax1.grid(axis='y', linestyle='--', alpha=0.3)
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 3, f'{yval:.1f}', ha='center', va='bottom', fontsize=9, color='#67e8f9')

# Plot 2: Smart Contract Gas Consumption
operations = ['registerFile (v1)', 'addVersion (v2)', 'authCollaborator', 'logVerification', 'revokeCollab', 'verifyHash (view)']
gas_units = [354375, 224845, 97970, 40479, 38417, 0]
colors = ['#3b82f6', '#6366f1', '#10b981', '#f59e0b', '#ec4899', '#14b8a6']

bars2 = ax2.barh(operations, gas_units, color=colors, alpha=0.85)
ax2.set_title('Smart Contract Gas Consumption Profile', fontsize=12, fontweight='bold', pad=12)
ax2.set_xlabel('Gas Units Consumed', fontsize=10)
ax2.grid(axis='x', linestyle='--', alpha=0.3)
for bar in bars2:
    xval = bar.get_width()
    label = f'{xval:,}' if xval > 0 else '0 (FREE)'
    ax2.text(xval + 8000, bar.get_y() + bar.get_height()/2, label, ha='left', va='center', fontsize=9, color='white')

plt.tight_layout()
plt.show()